# Our Accurate CO Count — Berkeley Housing Completions 2018-2026

**An independent certificate-of-occupancy count, computed from our reconstructed pipeline
(`berkeley_housing_v2.db`) alone — no city/CKAN figures enter this number.** This is the
"how we counted" notebook; a companion notebook reconciles it against the city's submitted APR.

_Read-only: this notebook queries the database, it does not modify it._

## Provenance — connect read-only and pin the database SHA

In [1]:
import sqlite3, hashlib
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
while not (ROOT/'databases'/'berkeley_housing_v2.db').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB = ROOT/'databases'/'berkeley_housing_v2.db'
print('canonical SHA-256:', hashlib.sha256(DB.read_bytes()).hexdigest()[:12])
conn = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)   # read-only
UC = (165,170,171,177)   # UC group-quarters projects, excluded from HCD unit totals

canonical SHA-256: dea44e4608c8


## Method — how a completion is defined and counted

Every number below follows one consistent rule set:

1. **Finaled = CO.** A building permit that reached its *Finaled* inspection is treated as a
   certificate of occupancy (de-facto completion). The completion is attributed to the **year of the
   Finaled date**.
2. **CKAN is a parcel-pointer, never a data source.** Where the city's APR (state CKAN portal) was used,
   it only identified *which parcels* the city listed as completing in a year. The **unit counts, dates,
   and coordinates all come from primary sources** — CPRA building-permit records + Alameda County
   assessor parcels. No figure here is copied from CKAN.
3. **Primary-permit-confirmed.** A completion is counted only when a Finaled ADU-construction or
   new-structural permit actually exists for that parcel/year. A CKAN listing alone does not assert a
   completion (this is stricter than simply trusting the city's list).
4. **ADU-aware classification.** Each permit is classified PRIMARY (a real housing milestone) vs
   SUBSIDIARY (solar, siding, window, demo, panel upgrade…). An ADU / JADU / conversion / legalization
   *is* a completion; an alteration on an existing-ADU parcel is *not*.
5. **Honest unknowns.** Fields the primary sources don't carry (bedroom mix, tenure, income) are stored
   as explicit *Unknown* with provenance — never inferred and never filled from CKAN.
6. **Group quarters excluded.** UC Berkeley student housing (dormitories) cannot count as HCD units and
   is excluded from the totals.

The single source of truth is the view **`v_projects_flat`**, which bakes in the permit classification
fix (subsidiary permits do not drive completion milestones).

## The count — net-new CO units by year, 2018-2026

In [2]:
years = range(2018, 2027)
rows = []
for y in years:
    r = conn.execute(
        'SELECT COUNT(*) p, COALESCE(SUM(total_units),0) u FROM v_projects_flat '
        f"WHERE substr(co_issued_date,1,4)='{y}' AND project_id NOT IN {UC}").fetchone()
    rows.append({'Year': y, 'Projects': r[0], 'Net-new CO units': r[1]})
counts = pd.DataFrame(rows).set_index('Year')
display(counts)
print('Total 2018-2026:', int(counts['Net-new CO units'].sum()), 'units across', int(counts['Projects'].sum()), 'projects')

,Projects,Net-new CO units
Year,,
2018,60,70
2019,96,98
2020,74,76
2021,103,107
2022,81,84
2023,107,631
2024,99,709
2025,100,531
2026,2,216


Total 2018-2026: 2522 units across 722 projects


## Composition — ADU / small vs multifamily

Split each year into **ADU/small (≤4 units)** and **multifamily (≥5 units)**. Note the scope boundary:
**2018-2022 currently hold only the ADU/small backfill** — the pre-policy *major* projects are a
separate, later verification pass — so those years are ADU-only by construction. 2023-2026 carry both.

In [3]:
rows = []
for y in years:
    rs = conn.execute('SELECT total_units u FROM v_projects_flat '
                      f"WHERE substr(co_issued_date,1,4)='{y}' AND project_id NOT IN {UC}").fetchall()
    u = [r[0] or 0 for r in rs]
    rows.append({'Year': y,
                 'ADU/small units': sum(x for x in u if x <= 4), 'ADU/small proj': sum(1 for x in u if x <= 4),
                 'Multifamily units': sum(x for x in u if x >= 5), 'Multifamily proj': sum(1 for x in u if x >= 5)})
split = pd.DataFrame(rows).set_index('Year')
display(split)

,ADU/small units,ADU/small proj,Multifamily units,Multifamily proj
Year,,,,
2018,70,60,0,0
2019,98,96,0,0
2020,76,74,0,0
2021,107,103,0,0
2022,84,81,0,0
2023,109,98,522,9
2024,103,93,606,6
2025,107,91,424,9
2026,0,0,216,2


## Primary-permit-confirmed basis

Every counted completion is backed by a PRIMARY-classified permit event in the database. The cell below
confirms that the counted projects carry a `permit_classified_primary` event and reports how many rely on
the *Finaled→CO inference* (a real Finaled date inferred to the CO milestone) — a transparency check, not
a defect.

In [4]:
counted = [r[0] for r in conn.execute('SELECT project_id FROM v_projects_flat '
           f"WHERE substr(co_issued_date,1,4) BETWEEN '2018' AND '2026' AND project_id NOT IN {UC}")]
ids = ','.join(map(str, counted))
prim = conn.execute(f'SELECT COUNT(DISTINCT project_id) FROM project_events WHERE event_type_id=26 AND project_id IN ({ids})').fetchone()[0]
inf  = conn.execute(f'SELECT COUNT(*) FROM project_events WHERE event_type_id=17 AND is_inferred=1 AND project_id IN ({ids})').fetchone()[0]
co   = conn.execute(f'SELECT COUNT(*) FROM project_events WHERE event_type_id=17 AND project_id IN ({ids})').fetchone()[0]
print(f'counted completions: {len(counted)} projects')
print(f'  with a PRIMARY-classified permit: {prim}')
print(f'  CO events that are Finaled->CO inferences: {inf} of {co}  (real dates, inferred milestone)')

counted completions: 722 projects
  with a PRIMARY-classified permit: 715
  CO events that are Finaled->CO inferences: 705 of 747  (real dates, inferred milestone)


## Honest unknowns & date-quality caveats

- **Tenure / income** are stored as *Unknown* for ADU completions (primary sources don't carry them); a
  small set of legacy major-project rows carry v1-migration affordability defaults that are flagged
  unsourced and must not be published as fact.
- **Dates** are reliable for *year attribution* but not for precise duration math: most CO events are
  Finaled→CO inferences, and a residue of legacy records carry Jan-1 placeholder dates (e.g. 2150
  Kittredge is correctly in CY2024 but stored as `2024-01-01` pending a Finaled-date correction to
  `2024-03-20`). These do not affect the per-year counts.
- **Coordinates**: a handful of parcels lack assessor coordinates (no map pin; the unit still counts).

---
*This count is independent of the city's APR. The companion reconciliation notebook compares it,
project by project, against the city's submitted figures.*

In [5]:
conn.close()